## Initialization
Run this block once to import all needed libraries and initialize classes.

In [ ]:
import uproot
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipyfilechooser import FileChooser


# [] Modify this class to work with Nika's numpy-based code
class Events:
    def __init__(self, event_tree, base_var):
        self.event_tree = event_tree    # The raw event tree from the root file
        self.base_var = base_var        # The base variable to be plotted. This will be 'cut' by other variables
        self._variables = []            # List of strings; the names of variables which will be used
        self.df = []                    # Pandas dataframe to contain the data from selected variables
        self.cut_mask = []              # Pandas boolean mask used to create a list of cut entries

        self._variables.append(base_var)

    @property
    def variables(self):
        return self._variables
    
    @variables.setter
    def variables(self, vars=[]):
        '''Takes in a list of cutter variables and appends them to variables'''
        self.variables.extend(vars)

    def create_dataframe(self):
        '''Creates a pandas dataframe from the list of variables'''
        self.df = self.event_tree.arrays(self.variables, library="pd")

        self.cut_mask = pd.Series(True, index=self.df.index)

    def reset_mask(self):
        self.cut_mask = pd.Series(True, index=self.df.index)

    def filter_rows_range(self, var, min, max):
        '''Takes in a variable name string and a min/max and removes any rows where that variable's value does not fall within the range'''
        if var not in self._variables:
            raise Exception(f"Error: Cutter variable {var} is not in list of variables")

        self.reset_mask()
        self.cut_mask &= self.df[var].between(min, max, inclusive="both")

    def filter_rows_bool(self, var, boolean):
        '''Takes in a variable name string and a true/false and removes any rows where that variable's value does not match the boolean'''
        if var not in self._variables:
            raise Exception(f"Error: Cutter variable {var} is not in list of variables")
        elif not isinstance(boolean, bool):
            raise TypeError("Error: input must be a boolean")
        
        self.cut_mask &= self.df[var] == boolean
        
    def is_bool_variable(self, var):
        '''Checks whether or not a cutter variable contains boolean data. If not, it will always contain float data instead.'''
        if var not in self._variables:
            raise Exception(f"Error: Cutter variable {var} is not in list of variables")
        
        for x in self.df[var].head(100):    # Checks the variable's first 100 entries to see if any of them are non-boolean. 100 might be overkill but whatever
            if not (x == 1 or x == 0):
                return False
        
        return True


## File selection
Run this block to select the root file you'd like to use in the following blocks.
You can change your file selection without needing to re-run this block.

In [2]:
fc = FileChooser("data/")
fc.filter_pattern = "*.root"

display(fc)

FileChooser(path='/Users/elliotlyons/Desktop/Classwork/CERN Project PRO1000/Project-116/data', filename='', ti…

## Cutting

In [6]:
print(f"Opening file: {fc.selected_filename}")
file = uproot.open(fc.selected)

print(file.keys())

keys = file.keys()[1]   
tree = file[keys]
print(tree.keys())

#reading branches from the list
all_branches = tree.keys()
variables = list(tree.keys())

###Conditions###
#Add new functions and limits here then change only the combine_masks function to add new cuts to the final plot
#rest should be unchanged

def kaon_probability_mask(k_threshold=0.85): # change the treshold if you want more or less stict cuts
    kplus = tree["Kplus_ProbNNk"].array(library="np")
    #kminus = tree["Kmin_ProbNNk"].array(library="np")
    #kstar = tree["kst_ProbNNk"].array(library="np")

    kplus_mask = np.isfinite(kplus) & (kplus > k_threshold)
    #kminus_mask = np.isfinite(kminus) & (kminus > k_threshold)
    #kstar_mask = np.isfinite(kstar) & (kstar > k_threshold)
    return kplus_mask 

def pminus_probability_mask(pmin_threshold=0.1): # change the treshold if you want more or less stict cuts
    pminus = tree["piminus_ProbNNpi"].array(library="np")
    pminus_mask = np.isfinite(pminus) & (pminus > pmin_threshold)
    return pminus_mask

#checking if it is not a muon
def mu_isMuon_mask():
    muplus_isMuon = tree["muplus_isMuon"].array(library="np")
    muplus_mask = np.isfinite(muplus_isMuon) & (muplus_isMuon != 0)

    muminus_isMuon = tree["muminus_isMuon"].array(library="np")
    muminus_mask = np.isfinite(muminus_isMuon) & (muminus_isMuon != 0)
    return muplus_mask & muminus_mask

def b0_flight_distance_mask(min_fdchi2=100):

    flight_chi2 = tree["B0_FDCHI2_OWNPV"].array(library="np")
    flight_distance_chi2_mask = np.isfinite(flight_chi2) & (flight_chi2 > min_fdchi2)
    return (flight_distance_chi2_mask)

#combine conditions -add new masks to variable list here
def combine_masks(k_threshold=0.85, pmin_threshold=0.1, min_fdchi2=100):
    return (kaon_probability_mask(k_threshold) 
            & pminus_probability_mask(pmin_threshold) 
            & mu_isMuon_mask() 
            & b0_flight_distance_mask(min_fdchi2))
       


###Plotting two graphs next to each other
def plot_with_cuts(variable, bins=100, x_min=None, x_max=10000):

    # Load raw data (never modify this)
    data = tree[variable].array(library="np")

    # Label
    if variable == "B0_PT":
        data_plot = data / 1000.0
        xlabel = "B0_PT [GeV/c]"
        plt.yscale("log")
    else:
        data_plot = data
        xlabel = variable

    # compute data for the right graph 
    mask = combine_masks()
    filtered_data = data_plot[mask]

    # Two graphs side by side
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

    axes[0].hist(data, bins=bins, range=(x_min, x_max), histtype="step")
    axes[0].set_title("Original")
    axes[0].set_xlabel(xlabel)
    axes[0].set_ylabel("Number of events")

    axes[1].hist(filtered_data, bins=bins, range=(x_min, x_max), histtype="step",color="orange")
    axes[1].set_title(f"After cuts: ")
    axes[1].set_xlabel(xlabel)
    axes[1].set_ylabel("Number of events")

    #plt.yscale("log")
    plt.tight_layout()
    plt.show()

    print("Original events:", len(data))
    print("Events after cut:", len(filtered_data))
    print("Removed events:", len(data) - len(filtered_data))


###Interactive part###
#choose variable to plot
variable_dropdown = widgets.Dropdown(
    options=variables,
    description="Variable:"
)
#sliders for cuts
bins_slider = widgets.IntSlider(
    value=100,
    min=1,
    max=300,
    step=1,
    description="Bins:"
)
k_threshold_slider = widgets.FloatSlider(
    value=0.85,
    min=0,
    max=1,
    step=0.01,
    description="K+ threshold:"
)
pmin_threshold_slider = widgets.FloatSlider(
    value=0.1,
    min=0,
    max=1,
    step=0.01,
    description="pi- threshold:"
)
fdchi2_threshold_slider = widgets.FloatSlider(
    value=100,
    min=0,
    max=500,
    step=1,
    description="Flight Distance CHI2 threshold:"
)
widgets.interact(
    plot_with_cuts,
    variable=variable_dropdown,
    k_threshold=k_threshold_slider,
    pmin_threshold=pmin_threshold_slider,
    min_fdchi2=fdchi2_threshold_slider,
    bins=bins_slider,
    x_min=widgets.FloatText(value=None, description="x min"),
    x_max=widgets.FloatText(value=None, description="x max")
)


Opening file: 00385270_00000001_1.dvntuple.root
['MyDecayTree_muons;1', 'MyDecayTree_muons/DecayTree;1', 'GetIntegratedLuminosity;1', 'GetIntegratedLuminosity/LumiTuple;1']
['B0_MCORR', 'B0_MCORRERR', 'B0_MCORRVERTEXERR', 'B0_PTREL', 'B0_ENDVERTEX_X', 'B0_ENDVERTEX_Y', 'B0_ENDVERTEX_Z', 'B0_ENDVERTEX_XERR', 'B0_ENDVERTEX_YERR', 'B0_ENDVERTEX_ZERR', 'B0_ENDVERTEX_CHI2', 'B0_ENDVERTEX_NDOF', 'B0_ENDVERTEX_COV_', 'B0_OWNPV_X', 'B0_OWNPV_Y', 'B0_OWNPV_Z', 'B0_OWNPV_XERR', 'B0_OWNPV_YERR', 'B0_OWNPV_ZERR', 'B0_OWNPV_CHI2', 'B0_OWNPV_NDOF', 'B0_OWNPV_COV_', 'B0_IP_OWNPV', 'B0_IPCHI2_OWNPV', 'B0_FD_OWNPV', 'B0_FDCHI2_OWNPV', 'B0_DIRA_OWNPV', 'B0_P', 'B0_PT', 'B0_PE', 'B0_PX', 'B0_PY', 'B0_PZ', 'B0_MM', 'B0_MMERR', 'B0_M', 'B0_ID', 'B0_TAU', 'B0_TAUERR', 'B0_TAUCHI2', 'B0_L0Global_Dec', 'B0_L0Global_TIS', 'B0_L0Global_TOS', 'B0_Hlt1Global_Dec', 'B0_Hlt1Global_TIS', 'B0_Hlt1Global_TOS', 'B0_Hlt1Phys_Dec', 'B0_Hlt1Phys_TIS', 'B0_Hlt1Phys_TOS', 'B0_Hlt2Global_Dec', 'B0_Hlt2Global_TIS', 'B0_Hlt2Gl

interactive(children=(Dropdown(description='Variable:', options=('B0_MCORR', 'B0_MCORRERR', 'B0_MCORRVERTEXERR…

<function __main__.plot_with_cuts(variable, bins=100, x_min=None, x_max=10000)>